# Project 1.1 - Semantic Similarity Search

## Concepts Covered

- Embeddings
- Word2Vec
- Semantic Similarity
- Vector Search
- Modern Embedding Models

## Goal

Build a semantic search engine that retrieves similar documents using embeddings.

## Questions We Will Answer

1. What is an embedding?
2. How does Word2Vec learn embeddings?
3. Why do similar words/documents have similar vectors?
4. How do modern embedding models work?
5. How is semantic search used in industry?

In [199]:
import sys

print(sys.executable)

c:\Users\ishik\anaconda3\envs\ai-from-scratch\python.exe


In [1]:
#Loading Dataset

import pandas as pd
import kagglehub

In [2]:
path = kagglehub.dataset_download(
    "rounakbanik/the-movies-dataset"
)

print(path)

C:\Users\ishik\.cache\kagglehub\datasets\rounakbanik\the-movies-dataset\versions\7


In [3]:
import os

files = os.listdir(path)

files

['credits.csv',
 'keywords.csv',
 'links.csv',
 'links_small.csv',
 'movies_metadata.csv',
 'ratings.csv',
 'ratings_small.csv']

In [4]:
movies_df1 = pd.read_csv(
    path + "/movies_metadata.csv",
    low_memory=False
)

movies_df1.head()

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0
3,False,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",...,1995-12-22,81452156.0,127.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Friends are the people who let you be yourself...,Waiting to Exhale,False,6.1,34.0
4,False,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",NaN,11862,tt0113041,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,...,1995-02-10,76578911.0,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,False,5.7,173.0


In [5]:
movies_df1.columns

Index(['adult', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id',
       'imdb_id', 'original_language', 'original_title', 'overview',
       'popularity', 'poster_path', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'video',
       'vote_average', 'vote_count'],
      dtype='str')

In [6]:
movies_df = movies_df1[
    ["title", "overview", "genres"]
].copy()

In [7]:
movies_df.shape

(45466, 3)

### Data Cleaning

In [8]:
movies_df["overview"].head(5)

0    Led by Woody, Andy's toys live happily in his ...
1    When siblings Judy and Peter discover an encha...
2    A family wedding reignites the ancient feud be...
3    Cheated on, mistreated and stepped on, the wom...
4    Just when George Banks has recovered from his ...
Name: overview, dtype: str

In [9]:
movies_df["overview"].isna().sum()

np.int64(954)

In [10]:
movies_df = movies_df.dropna(subset=["overview"])

In [11]:
movies_df.shape

(44512, 3)

In [12]:
sample_df= movies_df

In [13]:
import re

In [14]:
def preprocess(text):
    text=text.lower()
    text=re.sub(r"[^a-zA-Z\s]","",text)
    token=text.split()
    return token


In [15]:
#test
preprocess(
    "Andy's Toys Live Happily!"
)

['andys', 'toys', 'live', 'happily']

In [16]:
sample_df.shape

(44512, 3)

In [17]:
corpus = sample_df["overview"].apply(preprocess)

In [18]:
corpus.iloc[0]

['led',
 'by',
 'woody',
 'andys',
 'toys',
 'live',
 'happily',
 'in',
 'his',
 'room',
 'until',
 'andys',
 'birthday',
 'brings',
 'buzz',
 'lightyear',
 'onto',
 'the',
 'scene',
 'afraid',
 'of',
 'losing',
 'his',
 'place',
 'in',
 'andys',
 'heart',
 'woody',
 'plots',
 'against',
 'buzz',
 'but',
 'when',
 'circumstances',
 'separate',
 'buzz',
 'and',
 'woody',
 'from',
 'their',
 'owner',
 'the',
 'duo',
 'eventually',
 'learns',
 'to',
 'put',
 'aside',
 'their',
 'differences']

In [19]:
type(corpus.iloc[0])

list

In [20]:
corpus

0        [led, by, woody, andys, toys, live, happily, i...
1        [when, siblings, judy, and, peter, discover, a...
2        [a, family, wedding, reignites, the, ancient, ...
3        [cheated, on, mistreated, and, stepped, on, th...
4        [just, when, george, banks, has, recovered, fr...
                               ...                        
45461    [rising, and, falling, between, a, man, and, w...
45462    [an, artist, struggles, to, finish, his, work,...
45463    [when, one, of, her, hits, goes, wrong, a, pro...
45464    [in, a, small, town, live, two, brothers, one,...
45465    [years, after, decriminalisation, of, homosexu...
Name: overview, Length: 44512, dtype: object

In [21]:
from gensim.models import Word2Vec

model = Word2Vec(
    sentences=corpus,
    vector_size=100,
    window=5,
    min_count=5, #ignore a word if it comes fewer than 5 times
    sg=1   #skip gram
)


In [22]:
model

In [23]:
len(model.wv) #wv: word vector

23206

In [24]:
model.wv.index_to_key[:20] #word2vec sorts words by frequency

['the',
 'a',
 'and',
 'to',
 'of',
 'in',
 'is',
 'his',
 'with',
 'her',
 'he',
 'for',
 'on',
 'an',
 'by',
 'that',
 'as',
 'who',
 'their',
 'from']

In [25]:
model.wv["love"]

array([ 0.49845216, -0.0909086 , -0.56484556, -0.3936822 ,  0.24623048,
       -0.32080004,  0.7360908 ,  0.38906178, -0.02285904,  0.36587933,
        0.38049972, -0.34649986, -0.16444226, -0.30441415,  0.6316581 ,
        0.28292283,  0.08356967, -0.19286288,  0.1396041 , -0.2431731 ,
        0.23483087, -0.44939438,  0.33386818, -0.3039783 ,  0.139052  ,
        0.15662327,  0.09374507, -0.12753883, -0.20300674,  0.05266475,
        0.09286278, -0.15770441,  0.01944587,  0.07071733, -0.12295141,
        0.01970753,  0.13859631, -0.15242541, -0.38036755, -0.39394918,
        0.14952256, -0.26004615,  0.20441231, -0.63252366,  0.09508075,
       -0.5501449 , -0.69549173, -0.03855563, -0.5732031 ,  0.3196592 ,
       -0.03220778, -0.17586306, -0.01184905,  0.57425654, -0.13842231,
        0.23320703,  0.5630716 ,  0.01391863,  0.24684447,  0.36369658,
       -0.05570951, -0.44716677,  0.3596341 , -0.256349  , -0.2462391 ,
        0.29052958, -0.25298434,  0.23559035, -0.7019916 ,  0.31

In [26]:
model.wv.most_similar("love", topn=10)

[('madly', 0.6974092125892639),
 ('romance', 0.6295963525772095),
 ('passionately', 0.609203577041626),
 ('asleep', 0.6032971739768982),
 ('headoverheels', 0.5990219116210938),
 ('affection', 0.5927589535713196),
 ('romances', 0.588981568813324),
 ('savannah', 0.5824909806251526),
 ('jasmine', 0.5822441577911377),
 ('hunky', 0.581867516040802)]

In [27]:
model.wv["love"].shape

(100,)

In [28]:
model.wv.similarity("love", "relationship")

np.float32(0.4950326)

In [29]:
model.wv.similarity("love", "murder")

np.float32(0.2722658)

In [30]:
import numpy as np

def get_movie_embedding(tokens, model):

    vectors = []

    for word in tokens:

        if word in model.wv:
            vectors.append(model.wv[word])

    if len(vectors) == 0:
        return np.zeros(model.vector_size)

    return np.mean(vectors, axis=0)

In [31]:
movie_vector = get_movie_embedding(
    corpus.iloc[0],
    model
)

movie_vector.shape

(100,)

In [32]:
movie_embeddings = []

for tokens in corpus:
    movie_embeddings.append(
        get_movie_embedding(tokens, model)
    )

movie_embeddings = np.array(movie_embeddings)

movie_embeddings.shape

(44512, 100)

In [33]:
movie_embeddings[0] #100 size vector created for each movie

array([ 0.03472982,  0.1769689 , -0.06396186,  0.08598465,  0.04531553,
       -0.17569754,  0.05902098,  0.49715894, -0.10845191, -0.08682041,
       -0.01175932, -0.24982975,  0.06084479,  0.01012477,  0.18723468,
       -0.06868266,  0.09973083, -0.0918012 , -0.08284652, -0.37685615,
        0.09503317, -0.07144938,  0.1983397 , -0.06439611,  0.07919763,
        0.02891593, -0.14143261, -0.14105941, -0.19810353,  0.0493108 ,
        0.24539411, -0.11647174,  0.11439059, -0.15743865, -0.08852183,
        0.21517764,  0.08097596,  0.05148553, -0.12057899, -0.07274853,
        0.08552402, -0.12285481, -0.26739794,  0.11909049,  0.18789497,
       -0.16144866, -0.16175123, -0.06612971,  0.04228032,  0.11180358,
        0.03779549, -0.18741332, -0.01884172, -0.12896205, -0.01459491,
        0.04440758,  0.22011842, -0.10601693, -0.03686021,  0.06335481,
        0.01256718, -0.01083504,  0.24747856,  0.03627649, -0.28155991,
        0.17739137,  0.01851424,  0.16791859, -0.40591952,  0.09

In [34]:
sample_df.head()

,title,overview,genres
0,Toy Story,"Led by Woody, Andy's toys live happily in his ...","[{'id': 16, 'name': 'Animation'}, {'id': 35, '..."
1,Jumanji,When siblings Judy and Peter discover an encha...,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '..."
2,Grumpier Old Men,A family wedding reignites the ancient feud be...,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ..."
3,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...","[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam..."
4,Father of the Bride Part II,Just when George Banks has recovered from his ...,"[{'id': 35, 'name': 'Comedy'}]"


In [35]:
sample_df = sample_df.reset_index(drop=True)

In [36]:
sample_df.head()

,title,overview,genres
0,Toy Story,"Led by Woody, Andy's toys live happily in his ...","[{'id': 16, 'name': 'Animation'}, {'id': 35, '..."
1,Jumanji,When siblings Judy and Peter discover an encha...,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '..."
2,Grumpier Old Men,A family wedding reignites the ancient feud be...,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ..."
3,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...","[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam..."
4,Father of the Bride Part II,Just when George Banks has recovered from his ...,"[{'id': 35, 'name': 'Comedy'}]"


In [37]:
sample_df["embedding"] = list(movie_embeddings)

In [38]:
sample_df.head(2)

,title,overview,genres,embedding
0,Toy Story,"Led by Woody, Andy's toys live happily in his ...","[{'id': 16, 'name': 'Animation'}, {'id': 35, '...","[0.03472982347011566, 0.17696890234947205, -0...."
1,Jumanji,When siblings Judy and Peter discover an encha...,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...","[0.05683300644159317, 0.09012217819690704, -0...."


In [39]:
from sklearn.metrics.pairwise import cosine_similarity

In [40]:
vec1 = movie_embeddings[0]
vec2 = movie_embeddings[1]

In [41]:
cosine_similarity(
    vec1.reshape(1, -1),
    vec2.reshape(1, -1)
)

array([[0.93016475]])

In [42]:
sample_df.iloc[0]["title"]

'Toy Story'

In [43]:
sample_df.iloc[1]["title"]

'Jumanji'

In [44]:
def find_movie_index(title):

    matches = sample_df[
        sample_df["title"].str.lower() == title.lower()
    ]

    if len(matches) == 0:
        return None

    return matches.index[0]

In [45]:
find_movie_index("Troy")

7267

In [46]:
def find_similar_movies(title, top_n=5):

    movie_idx = find_movie_index(title)

    if movie_idx is None:
        print("Movie not found")
        return

    query_embedding = movie_embeddings[movie_idx]

    similarities = cosine_similarity(
        query_embedding.reshape(1, -1),
        movie_embeddings
    )[0]

    similar_indices = similarities.argsort()[::-1]

    for idx in similar_indices[1:top_n+1]:
        print(
            sample_df.iloc[idx]["title"],
            "->",
            round(similarities[idx], 3)
        )

In [47]:
find_similar_movies("Troy")

Helen of Troy -> 0.982
Asterix & Obelix: Mission Cleopatra -> 0.981
You Won't Have Alsace-Lorraine -> 0.981
Marketa Lazarová -> 0.98
East/West -> 0.979


In [48]:
find_similar_movies("Toy Story")

Soul Food -> 0.973
Blackadder Back & Forth -> 0.971
Four Mothers -> 0.971
Women Vs Men -> 0.97
How to Win at Checkers (Every Time) -> 0.97


In [49]:
find_similar_movies("Jumanji")

Cookers -> 0.975
To Boldly Flee -> 0.973
BreadCrumbs -> 0.973
Breathing Room -> 0.973
An American Terror -> 0.972


In [50]:
find_similar_movies("Titanic")

A Talking Picture -> 0.976
What About Dick? -> 0.972
The Color Purple -> 0.972
Four Nights of a Dreamer -> 0.972
Dischord -> 0.972


### Evaluating the model

In [51]:
movies_df.columns

Index(['title', 'overview', 'genres'], dtype='str')

In [52]:
movies_df["genres"].iloc[0]

"[{'id': 16, 'name': 'Animation'}, {'id': 35, 'name': 'Comedy'}, {'id': 10751, 'name': 'Family'}]"

In [53]:
import ast

def extract_genres(genre_str):

    try:
        genres = ast.literal_eval(genre_str)
        return [g["name"] for g in genres]

    except:
        return []

In [54]:
movies_df["genre_list"] = movies_df["genres"].apply(extract_genres)

In [55]:
movies_df[["title", "genre_list"]].head()

,title,genre_list
0,Toy Story,"[Animation, Comedy, Family]"
1,Jumanji,"[Adventure, Fantasy, Family]"
2,Grumpier Old Men,"[Romance, Comedy]"
3,Waiting to Exhale,"[Comedy, Drama, Romance]"
4,Father of the Bride Part II,[Comedy]


In [56]:
def genre_match(movie1_idx, movie2_idx):

    genres1 = set(movies_df.iloc[movie1_idx]["genre_list"])
    genres2 = set(movies_df.iloc[movie2_idx]["genre_list"])

    return len(genres1.intersection(genres2)) > 0

In [57]:
genre_match(0, 1)

True

In [58]:
def get_similar_movie_indices(title, top_n=5):

    movie_idx = find_movie_index(title)

    query_embedding = movie_embeddings[movie_idx]

    similarities = cosine_similarity(
        query_embedding.reshape(1, -1),
        movie_embeddings
    )[0]

    similar_indices = similarities.argsort()[::-1]

    return similar_indices[1:top_n+1]

In [59]:
get_similar_movie_indices("Troy")

array([ 7732,  9233, 34085, 12486,  3221])

In [60]:
indices = get_similar_movie_indices("Troy")

for idx in indices:
    print(
        movies_df.iloc[idx]["title"],
        movies_df.iloc[idx]["genre_list"]
    )

Helen of Troy ['Action', 'Adventure', 'Drama', 'Romance', 'War']
Asterix & Obelix: Mission Cleopatra ['Family', 'Fantasy', 'Comedy', 'Adventure']
You Won't Have Alsace-Lorraine ['Comedy']
Marketa Lazarová ['Drama', 'History']
East/West ['Drama']


In [61]:
movies_df[
    movies_df["title"].str.lower() == "troy"
]["genre_list"]

7291    [Adventure, Drama, War]
Name: genre_list, dtype: object

In [62]:
def genre_match_rate(title, top_n=5):

    movie_idx = find_movie_index(title)

    query_genres = set(
        movies_df.iloc[movie_idx]["genre_list"]
    )

    indices = get_similar_movie_indices(title, top_n)

    matches = 0

    for idx in indices:

        rec_genres = set(
            movies_df.iloc[idx]["genre_list"]
        )

        if len(query_genres.intersection(rec_genres)) > 0:
            matches += 1

    return matches / top_n

In [63]:
genre_match_rate("Troy")

0.8

In [64]:
sample_titles = (
    movies_df["title"]
    .dropna()
    .sample(1000, random_state=42)
)

In [65]:
scores = []

for movie in sample_titles:

    try:
        scores.append(
            genre_match_rate(movie)
        )

    except:
        pass

In [66]:
np.mean(scores)

np.float64(0.5834000000000001)